# Подход к решению задачи классификации ботов

Задача состоит в классификации кук на предмет автоматизированного сбора данных на основе событий внутри суточного окна

### 1. Подготовка данных и фильтрация по окну

События объединяются с метаданными куки и фильтруются по временному окну от window_start_ts до window_end_ts

События вне интервала наблюдения отсекаются для исключения утечки информации из будущего

### 2. Извлечение признаков

На основе истории действий куки рассчитываются поведенческие и технические маркеры

* Временные интервалы: дельты времени между кликами, среднее, медиана, стандартное отклонение, межквартильный размах, коэффициент вариации и доли быстрых действий менее 100 миллисекунд, 200 миллисекунд, 500 миллисекунд, одной секунды и трех секунд
* Суточный ритм: активность в ночные и утренние часы, число уникальных активных часов за сутки
* Движение курсора: наличие координат мыши, число уникальных точек, среднее и максимальное расстояние перемещения курсора между соседними действиями
* Взаимодействие с каталогом: число уникальных объявлений, категорий, городов и запросов, глубина просмотра страниц выдачи, флаги фокуса на единственной категории или локации
* Соотношения типов действий: отношение просмотров объявлений к открытию контактов, отношение свайпов фото к просмотрам объявлений, доля запросов контактов и показов капчи
* Технические параметры: маркеры браузеров без интерфейса и скриптов автоматизации в User-Agent, операционные системы, длина строки агента и число платформ
* Метаданные: возраст куки от даты создания до окна наблюдения, флаг свежей куки, задержка до первого действия и покрытие окна активностью

### 3. Валидация

Тестовые данные расположены позже обучающей выборки по времени

Для локальной оценки используется сплит по времени начала окна наблюдения

Первые 80 процентов данных используются для обучения, а последние 20 процентов для валидации

Качество оценивается официальной метрикой precision_at_recall при целевой полноте не менее 0.70

### 4. Обучение моделей и ансамбль

Для финального сабмита применяется кросс-валидация на 5 фолдов со стратификацией по целевой переменной

На каждом фолде обучаются модели LightGBM и CatBoost

Предсказания усредняются по фолдам, переводятся в ранги и объединяются с весами 0.35 для LightGBM и 0.65 для CatBoost

Ранговое взвешивание дает стабильный порядок кук для целевой метрики

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from metric import precision_at_recall

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# функция генерации комплексного вектора признаков по событиям окна наблюдения
def extract_features(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    print(f"Фильтрация событий по окну наблюдения для {len(meta)} кук")
    # объединение событий с метаданными и отсечение записей вне интервала окна
    ev = events.merge(
        meta[["cookie_id", "cookie_created_at", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner"
    )
    ev = ev[(ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])].copy()

    # сортировка по времени для вычисления межсобытийных дельт
    ev.sort_values(by=["cookie_id", "event_ts"], inplace=True)

    print("Расчет интервалов времени и детекция технических аномалий")
    # дельта времени между соседними действиями куки в секундах
    ev["dt_prev"] = ev.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()

    # флаги неестественно быстрых кликов
    ev["is_sub_100ms"] = (ev["dt_prev"] < 0.1).astype(float)
    ev["is_sub_200ms"] = (ev["dt_prev"] < 0.2).astype(float)
    ev["is_sub_500ms"] = (ev["dt_prev"] < 0.5).astype(float)
    ev["is_subsecond"] = (ev["dt_prev"] < 1.0).astype(float)
    ev["is_sub_3s"] = (ev["dt_prev"] < 3.0).astype(float)

    # время суток для анализа ночной активности
    ev["hour"] = ev["event_ts"].dt.hour
    ev["is_night"] = ((ev["hour"] >= 1) & (ev["hour"] <= 5)).astype(float)
    ev["is_morning"] = ((ev["hour"] >= 6) & (ev["hour"] <= 11)).astype(float)

    # расстояние перемещения курсора мыши между соседними событиями
    ev["dx"] = ev.groupby("cookie_id")["pointer_x"].diff()
    ev["dy"] = ev.groupby("cookie_id")["pointer_y"].diff()
    ev["pointer_dist"] = np.sqrt(ev["dx"] ** 2 + ev["dy"] ** 2)

    # нормализация платформы
    ev["platform_clean"] = ev["platform"].fillna("").astype(str).str.lower()
    ev["is_android"] = (ev["platform_clean"] == "android").astype(float)
    ev["is_ios"] = (ev["platform_clean"] == "ios").astype(float)
    ev["is_web"] = ev["platform_clean"].isin(["web", "desktop"]).astype(float)

    # сигналы автоматизации и библиотек скрапинга в строке агента
    ua_str = ev["user_agent"].fillna("").astype(str)
    ev["is_headless"] = ua_str.str.contains("headless|phantom|selenium|puppeteer|playwright", case=False).astype(int)
    ev["is_bot_client"] = ua_str.str.contains("python|aiohttp|requests|curl|wget|bot|crawler|spider|scrapy|httpx", case=False).astype(int)
    ev["is_okhttp"] = ua_str.str.contains("okhttp", case=False).astype(int)
    ev["is_linux_desktop"] = (ua_str.str.contains("linux", case=False) & ~ua_str.str.contains("android", case=False)).astype(int)
    ev["is_firefox"] = ua_str.str.contains("firefox|fxios", case=False).astype(int)
    ev["is_yabrowser"] = ua_str.str.contains("yabrowser", case=False).astype(int)
    ev["ua_len"] = ua_str.str.len()

    # признаки целевых продавцов и поисковых запросов
    ev["is_private_seller"] = (ev["seller_type"] == "private").astype(float)
    ev["is_pro_seller"] = (ev["seller_type"] == "pro").astype(float)
    ev["has_query"] = (ev["search_query"].notna() & (ev["search_query"].astype(str).str.strip() != "")).astype(float)
    ev["has_pointer"] = (ev["pointer_x"].notna() & ev["pointer_y"].notna()).astype(int)
    ev["is_empty_query_search"] = ((ev["event_name"] == "search_results_view") & (ev["has_query"] == 0)).astype(float)

    print("Агрегация поведенческих признаков по cookie_id")
    grouped = ev.groupby("cookie_id")

    # базовые объемы сессии и темп кликов
    feats = pd.DataFrame(index=meta["cookie_id"].unique())
    feats["n_events"] = grouped.size()
    feats["first_event_ts"] = grouped["event_ts"].min()
    feats["last_event_ts"] = grouped["event_ts"].max()
    feats["session_duration_sec"] = (feats["last_event_ts"] - feats["first_event_ts"]).dt.total_seconds().clip(lower=0)
    feats["events_per_sec"] = feats["n_events"] / (feats["session_duration_sec"] + 1.0)
    feats["items_per_min"] = grouped["item_id"].nunique() / (feats["session_duration_sec"] / 60.0 + 1.0)

    # распределение межсобытийных дельт времени
    dt_stats = grouped["dt_prev"].agg(
        dt_mean="mean",
        dt_std="std",
        dt_median="median",
        dt_min="min",
        dt_max="max"
    )
    feats = feats.join(dt_stats)
    feats["dt_cv"] = feats["dt_std"] / (feats["dt_mean"] + 1e-4)

    dt_q25 = grouped["dt_prev"].quantile(0.25)
    dt_q75 = grouped["dt_prev"].quantile(0.75)
    feats["dt_iqr"] = dt_q75 - dt_q25

    # доли быстрых действий
    feats["sub_100ms_ratio"] = grouped["is_sub_100ms"].mean()
    feats["sub_200ms_ratio"] = grouped["is_sub_200ms"].mean()
    feats["sub_500ms_ratio"] = grouped["is_sub_500ms"].mean()
    feats["subsecond_ratio"] = grouped["is_subsecond"].mean()
    feats["sub_3s_ratio"] = grouped["is_sub_3s"].mean()

    # ночная и утренняя активность
    feats["night_event_ratio"] = grouped["is_night"].mean()
    feats["morning_event_ratio"] = grouped["is_morning"].mean()
    feats["active_hours_count"] = grouped["hour"].nunique()

    # динамика курсора мыши
    feats["has_pointer_ratio"] = grouped["has_pointer"].mean()
    feats["pointer_x_nunique"] = grouped["pointer_x"].nunique()
    feats["pointer_y_nunique"] = grouped["pointer_y"].nunique()
    feats["pointer_dist_mean"] = grouped["pointer_dist"].mean()
    feats["pointer_dist_std"] = grouped["pointer_dist"].std()
    feats["pointer_dist_max"] = grouped["pointer_dist"].max()
    feats["has_pointer_movement"] = ((feats["pointer_dist_max"] > 0) & feats["pointer_dist_max"].notna()).astype(float)

    # разнообразие сущностей в каталоге
    feats["item_nunique"] = grouped["item_id"].nunique()
    feats["category_nunique"] = grouped["item_category"].nunique()
    feats["location_nunique"] = grouped["item_location"].nunique()
    feats["search_query_nunique"] = grouped["search_query"].nunique()
    feats["event_name_nunique"] = grouped["event_name"].nunique()

    feats["item_per_event_ratio"] = feats["item_nunique"] / (feats["n_events"] + 1e-4)
    feats["events_per_item"] = feats["n_events"] / (feats["item_nunique"] + 1e-4)
    feats["items_per_category"] = feats["item_nunique"] / (feats["category_nunique"] + 1e-4)
    feats["categories_per_event"] = feats["category_nunique"] / (feats["n_events"] + 1e-4)
    feats["locations_per_event"] = feats["location_nunique"] / (feats["n_events"] + 1e-4)
    feats["is_single_category"] = (feats["category_nunique"] <= 1).astype(float)
    feats["is_single_location"] = (feats["location_nunique"] <= 1).astype(float)

    # глубина страниц поисковой выдачи
    search_page_stats = grouped["search_page"].agg(
        search_page_max="max",
        search_page_mean="mean",
        search_page_nunique="nunique"
    )
    feats = feats.join(search_page_stats)
    feats["search_page_deep_ratio"] = (ev["search_page"] > 3).groupby(ev["cookie_id"]).mean()

    # поведение с продавцами и поиском
    feats["private_seller_ratio"] = grouped["is_private_seller"].mean()
    feats["pro_seller_ratio"] = grouped["is_pro_seller"].mean()
    feats["query_event_ratio"] = grouped["has_query"].mean()
    feats["empty_query_search_ratio"] = grouped["is_empty_query_search"].mean()

    # доли используемых платформ
    feats["android_ratio"] = grouped["is_android"].mean()
    feats["ios_ratio"] = grouped["is_ios"].mean()
    feats["web_ratio"] = grouped["is_web"].mean()
    feats["platform_nunique"] = grouped["platform_clean"].nunique()

    # технические признаки браузера
    feats["is_headless_max"] = grouped["is_headless"].max()
    feats["is_bot_client_max"] = grouped["is_bot_client"].max()
    feats["is_linux_desktop_max"] = grouped["is_linux_desktop"].max()
    feats["is_firefox_max"] = grouped["is_firefox"].max()
    feats["is_yabrowser_max"] = grouped["is_yabrowser"].max()
    feats["is_okhttp_ratio"] = grouped["is_okhttp"].mean()
    feats["ua_len_mean"] = grouped["ua_len"].mean()
    feats["ua_nunique"] = grouped["user_agent"].nunique()

    # счетчики и доли каждого типа действия
    event_counts = ev.pivot_table(index="cookie_id", columns="event_name", aggfunc="size", fill_value=0)
    for col in event_counts.columns:
        feats[f"cnt_{col}"] = event_counts[col]
        feats[f"ratio_{col}"] = event_counts[col] / (feats["n_events"] + 1e-4)

    item_views = feats["cnt_item_view"] if "cnt_item_view" in feats.columns else 0
    photo_swipes = feats["cnt_photo_swipe"] if "cnt_photo_swipe" in feats.columns else 0
    phone_shows = feats["cnt_contact_phone_show"] if "cnt_contact_phone_show" in feats.columns else 0
    chat_opens = feats["cnt_contact_chat_open"] if "cnt_contact_chat_open" in feats.columns else 0
    captchas = feats["cnt_captcha_shown"] if "cnt_captcha_shown" in feats.columns else 0
    search_views = feats["cnt_search_results_view"] if "cnt_search_results_view" in feats.columns else 0

    # соотношения целевых конверсий
    feats["photo_per_item_ratio"] = photo_swipes / (item_views + 1.0)
    feats["contact_per_item_ratio"] = (phone_shows + chat_opens) / (item_views + 1.0)
    feats["captcha_event_ratio"] = captchas / (feats["n_events"] + 1e-4)
    feats["phone_per_event_ratio"] = phone_shows / (feats["n_events"] + 1e-4)
    feats["phone_to_search_ratio"] = phone_shows / (search_views + 1.0)
    feats["item_per_search_ratio"] = item_views / (search_views + 1.0)

    # метаданные куки и тайминги сессии
    feats = meta.set_index("cookie_id")[["cookie_created_at", "window_start_ts", "window_end_ts"]].join(feats)
    feats["cookie_age_days"] = (feats["window_start_ts"] - feats["cookie_created_at"]).dt.total_seconds() / 86400.0
    feats["cookie_is_fresh"] = (feats["cookie_age_days"] <= 1.0).astype(float)
    feats["time_to_first_event_sec"] = (feats["first_event_ts"] - feats["window_start_ts"]).dt.total_seconds().clip(lower=0)
    feats["time_from_last_event_sec"] = (feats["window_end_ts"] - feats["last_event_ts"]).dt.total_seconds().clip(lower=0)
    feats["active_window_coverage"] = feats["session_duration_sec"] / 86400.0

    cols_to_drop = ["first_event_ts", "last_event_ts", "cookie_created_at", "window_start_ts", "window_end_ts"]
    feats.drop(columns=[c for c in cols_to_drop if c in feats.columns], inplace=True)

    # обработка кук без событий внутри окна наблюдения
    feats["n_events"] = feats["n_events"].fillna(0)
    feats.fillna(-1, inplace=True)

    return meta[["cookie_id"]].merge(feats.reset_index(), on="cookie_id", how="left")

# загрузка данных и валидация пайплайна
def main():
    data_dir = Path("data")
    if not (data_dir / "train.csv").exists():
        data_dir = Path(".")

    print("Загрузка данных")
    train = pd.read_csv(data_dir / "train.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
    test = pd.read_csv(data_dir / "test.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
    events = pd.read_csv(data_dir / "events.csv.gz", parse_dates=["event_ts"])

    print(f"Размер train: {train.shape}, test: {test.shape}, events: {events.shape}")
    print(f"Доля ботов в train: {train.target.mean():.4f}")

    # расчет признаков для обучающей и тестовой выборок
    X_train_df = extract_features(events, train)
    X_test_df = extract_features(events, test)

    feature_cols = [c for c in X_train_df.columns if c not in ["cookie_id", "target"]]
    y_train = train["target"].values

    # разделение обучающей выборки по времени для локальной валидации
    split_date = train["window_start_ts"].quantile(0.80)
    val_mask = train["window_start_ts"] >= split_date
    tr_mask = ~val_mask

    print(f"\nВалидация по времени с порогом даты {split_date}")
    print(f"Строк в обучении: {tr_mask.sum()}, строк в валидации: {val_mask.sum()}")

    X_tr_val = X_train_df.loc[tr_mask, feature_cols]
    y_tr_val = y_train[tr_mask]
    X_va_val = X_train_df.loc[val_mask, feature_cols]
    y_va_val = y_train[val_mask]

    # обучение модели LightGBM на валидационном сплите
    lgb_val = LGBMClassifier(
        n_estimators=900,
        learning_rate=0.025,
        num_leaves=28,
        min_child_samples=40,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=1.0,
        reg_lambda=3.0,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )
    lgb_val.fit(X_tr_val, y_tr_val)
    preds_lgb_val = lgb_val.predict_proba(X_va_val)[:, 1]

    # обучение модели CatBoost на валидационном сплите
    cb_val = CatBoostClassifier(
        iterations=1100,
        learning_rate=0.025,
        depth=6,
        l2_leaf_reg=5,
        random_seed=SEED,
        verbose=0,
        thread_count=-1
    )
    cb_val.fit(X_tr_val, y_tr_val)
    preds_cb_val = cb_val.predict_proba(X_va_val)[:, 1]

    # ранговое усреднение вероятностей моделей
    rank_lgb_val = (rankdata(preds_lgb_val) - 1.0) / (len(preds_lgb_val) - 1.0)
    rank_cb_val = (rankdata(preds_cb_val) - 1.0) / (len(preds_cb_val) - 1.0)
    preds_ensemble_val = 0.35 * rank_lgb_val + 0.65 * rank_cb_val

    # расчет официальной метрики соревнований
    p_at_r_lgb = precision_at_recall(y_va_val, preds_lgb_val)
    p_at_r_cb = precision_at_recall(y_va_val, preds_cb_val)
    p_at_r_ens = precision_at_recall(y_va_val, preds_ensemble_val)

    print(f"LightGBM precision at recall 0.7: {p_at_r_lgb:.4f}")
    print(f"CatBoost precision at recall 0.7: {p_at_r_cb:.4f}")
    print(f"Ансамбль precision at recall 0.7: {p_at_r_ens:.4f}")

    # оценка важности признаков
    importances = pd.Series(lgb_val.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print("\nТоп 10 информативных признаков LightGBM:")
    print(importances.head(10))

    print("\nОбучение финального ансамбля на полной обучающей выборке")
    # кросс-валидация на 5 фолдов со стратификацией для финального ансамбля
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    test_preds_lgb = np.zeros(len(test))
    test_preds_cb = np.zeros(len(test))

    for fold, (train_idx, _) in enumerate(skf.split(X_train_df[feature_cols], y_train)):
        X_fold_tr = X_train_df.loc[train_idx, feature_cols]
        y_fold_tr = y_train[train_idx]

        model_lgb = LGBMClassifier(
            n_estimators=900,
            learning_rate=0.025,
            num_leaves=28,
            min_child_samples=40,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=SEED + fold,
            n_jobs=-1,
            verbose=-1
        )
        model_lgb.fit(X_fold_tr, y_fold_tr)
        test_preds_lgb += model_lgb.predict_proba(X_test_df[feature_cols])[:, 1] / 5.0

        model_cb = CatBoostClassifier(
            iterations=1100,
            learning_rate=0.025,
            depth=6,
            l2_leaf_reg=5,
            random_seed=SEED + fold,
            verbose=0,
            thread_count=-1
        )
        model_cb.fit(X_fold_tr, y_fold_tr)
        test_preds_cb += model_cb.predict_proba(X_test_df[feature_cols])[:, 1] / 5.0

    # ранговое усреднение финальных прогнозов для теста
    rank_lgb_test = (rankdata(test_preds_lgb) - 1.0) / (len(test_preds_lgb) - 1.0)
    rank_cb_test = (rankdata(test_preds_cb) - 1.0) / (len(test_preds_cb) - 1.0)
    final_test_score = 0.35 * rank_lgb_test + 0.65 * rank_cb_test

    # формирование файла сабмита
    submission = pd.DataFrame({
        "cookie_id": test["cookie_id"],
        "score": final_test_score
    })

    # проверки корректности формата файла
    assert len(submission) == len(test), f"Ожидалось {len(test)} строк"
    assert submission["score"].between(0, 1).all(), "Значения score вне диапазона от 0 до 1"
    assert submission["cookie_id"].equals(test["cookie_id"]), "Нарушен исходный порядок cookie_id"
    assert not submission.isna().any().any(), "Обнаружены пропуски в предсказаниях"

    submission.to_csv("submission.csv", index=False)
    print("\nФайл submission.csv успешно сформирован")
    print(submission.head(5))


if __name__ == "__main__":
    main()

Загрузка данных
Размер train: (11091, 5), test: (4909, 4), events: (328905, 14)
Доля ботов в train: 0.0811
Фильтрация событий по окну наблюдения для 11091 кук
Расчет интервалов времени и детекция технических аномалий
Агрегация поведенческих признаков по cookie_id
Фильтрация событий по окну наблюдения для 4909 кук
Расчет интервалов времени и детекция технических аномалий
Агрегация поведенческих признаков по cookie_id

Валидация по времени с порогом даты 2026-04-16 00:00:00
Строк в обучении: 8400, строк в валидации: 2691
LightGBM precision at recall 0.7: 0.6039
CatBoost precision at recall 0.7: 0.6667
Ансамбль precision at recall 0.7: 0.6432

Топ 10 информативных признаков LightGBM:
dt_median                   1387
cookie_age_days             1097
dt_iqr                       965
locations_per_event          955
time_to_first_event_sec      736
time_from_last_event_sec     713
dt_cv                        711
dt_min                       690
private_seller_ratio         668
search_page_m